**Class Conditioning and Generation**

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import os

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

In [2]:
PROCESSED_DIR = '../data/processed'

X_train = pd.read_csv(f'{PROCESSED_DIR}/X_train.csv')
y_train = pd.read_csv(f'{PROCESSED_DIR}/y_train.csv').squeeze()

print(f'X_train: {X_train.shape}')
print(f'Minority class (readmitted <30): {y_train.sum():,} samples')

X_train: (79472, 42)
Minority class (readmitted <30): 9,051 samples


In [3]:
scaler = joblib.load('../models/scaler.pkl')
NUMERIC_COLS = scaler.feature_names_in_.tolist()
print(NUMERIC_COLS)

['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age']


In [4]:
label_encoders = joblib.load('../models/label_encoders.pkl')

METADATA = {
    'n_features': X_train.shape[1],
    'numeric_cols': NUMERIC_COLS,
    'categorical_cols': {
        col: {'cardinality': len(le.classes_)}
        for col, le in label_encoders.items()
    }
}

print("Metadata:")
print(f"  Total features : {METADATA['n_features']}")
print(f"  Numeric cols   : {len(METADATA['numeric_cols'])}")
print(f"  Categorical cols: {METADATA['categorical_cols']}")

Metadata:
  Total features : 42
  Numeric cols   : 9
  Categorical cols: {'race': {'cardinality': 5}, 'diag_1': {'cardinality': 9}, 'diag_2': {'cardinality': 9}, 'diag_3': {'cardinality': 9}}


In [5]:
class ClassEmbedding(nn.Module):
    def __init__(self, num_classes: int, embed_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, embed_dim)

    def forward(self, labels: torch.Tensor) -> torch.Tensor:
        # labels: (batch,) int tensor
        return self.embedding(labels)  # → (batch, embed_dim)

# Test it standalone
embed_dim = 128  # agree this value with Jiten
class_embed = ClassEmbedding(num_classes=2, embed_dim=embed_dim)

test_labels = torch.tensor([0, 1, 1, 0])
out = class_embed(test_labels)
print(f"Input labels : {test_labels}")
print(f"Output shape : {out.shape}")  # should be (4, 128)
print("ClassEmbedding working correctly!")

Input labels : tensor([0, 1, 1, 0])
Output shape : torch.Size([4, 128])
ClassEmbedding working correctly!


In [7]:
def generate_synthetic_samples(model, class_embed, n_samples, label, T, device='cpu'):
    """
    Reverse diffusion loop — generates synthetic samples conditioned on a class label.
    
    Args:
        model       : Jiten's trained network (denoise_step method expected)
        class_embed : ClassEmbedding module
        n_samples   : how many synthetic rows to generate
        label       : 0 or 1 (1 = readmitted <30 days)
        T           : total diffusion timesteps (from Anum)
        device      : cpu or cuda
    """
    model.eval()
    with torch.no_grad():
        # Start from pure noise
        x = torch.randn(n_samples, METADATA['n_features']).to(device)
        y = torch.full((n_samples,), label, dtype=torch.long).to(device)
        c = class_embed(y)  # conditioning vector → (n_samples, embed_dim)

        # Reverse diffusion: T → 0
        for t in reversed(range(T)):
            t_tensor = torch.full((n_samples,), t, dtype=torch.long).to(device)
            # TODO: replace with Jiten's actual function signature
            x = model.denoise_step(x, t_tensor, c)

    return x.cpu().numpy()

print("Generation function defined.")

Generation function defined.
